In [3]:
import torch
import torch.nn as nn

#Experimentation with ANFIS layers



##1. Fuzzification Layer


###1.1 Proposal 1
To initialize it:
* data_shape: Shape of a batch of data
* features : Number of initial features (default = 1)
* mf: Function to be used as MemberShip Function. This function must be written in such a way that it receives a tensor as parameters (default = gaussian_3, Gaussian function with 3 parameters)
* params : List with the names of the MF parameters to use (default = ['sigma', 'mu', 'f'])
* init_mode: Premises initialization mode (default = 0)
  
The idea is that the layer stores the premises in a tensor that will have as an attribute, this implies that the mf that you want to apply must be written in a special way: 2 parameters, x (matrix of input data) and p (matrix of parameters to be used for each input data).

In [4]:
def gaussian(x, mu, sigma, f=1):
    return torch.exp( -f * torch.pow((x - mu), 2) / (2*torch.pow(sigma, 2)) )

In [5]:
def gaussian_3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [6]:
unic_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [7]:
def gaussian_2(x, p):
    return torch.exp(-torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [8]:
params=['sigma', 'mu', 'f']
features = 3
shape = x_multi.shape
shape

torch.Size([3, 2])

In [9]:
x_multi.unsqueeze(2).repeat(1, 1, features)

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[2, 2, 2],
         [4, 4, 4]],

        [[3, 3, 3],
         [6, 6, 6]]])

In [10]:
premises = torch.rand(shape[1], features, len(params))
#                    (       2,        3,           3)
premises

tensor([[[0.3246, 0.7322, 0.2297],
         [0.0981, 0.9386, 0.0558],
         [0.7075, 0.9492, 0.6726]],

        [[0.1451, 0.8692, 0.3247],
         [0.8726, 0.1143, 0.9101],
         [0.5765, 0.1411, 0.2319]]])

In [11]:
y = gaussian_3(x_multi.unsqueeze(2).repeat(1, 1, features), premises)
y

tensor([[[9.0692e-01, 9.7459e-01, 9.6857e-01],
         [4.7742e-01, 5.9572e-20, 7.5400e-06]],

        [[5.4814e-01, 8.9184e-01, 5.3604e-01],
         [4.1035e-02, 0.0000e+00, 2.3551e-30]],

        [[2.1586e-01, 7.6607e-01, 1.4062e-01],
         [6.3210e-04, 0.0000e+00, 0.0000e+00]]])

In [12]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Properties:
        data_shape : dimension of the input data
        features : number of initial features
        mf: function used as membership function
        params: list with parameter names of the mf
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, features=3, mf=gaussian_3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.features = features
        self.data_shape = data_shape
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(data_shape[1], features, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(data_shape[1], features, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(2).repeat(1, 1, self.features), self.premises)

    def show_premises(self):
        print("Premises tensor:")
        print(self.premises)

    def premises(self):
        return self.premises

In [13]:
train_data = torch.rand(5, 2)
train_data #5 2-dimensional input vectors (batch size = 5)

tensor([[0.2105, 0.8656],
        [0.5940, 0.8380],
        [0.9710, 0.7050],
        [0.5111, 0.9146],
        [0.3231, 0.9400]])

In [14]:
fuzzify_layer = FuzzifyLayer(train_data.shape, features=3)

In [15]:
'''
x1 premises
[[[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]],  (feature 3)

x2 premises
 [[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]]]  (feature 3)
'''
fuzzify_layer.show_premises()

Premises tensor:
tensor([[[0.9662, 0.8149, 0.4946],
         [0.1833, 0.0114, 0.1615],
         [0.9501, 0.0580, 0.8212]],

        [[0.8003, 0.1145, 0.7442],
         [0.3614, 0.1365, 0.4214],
         [0.3900, 0.7473, 0.4127]]])


In [16]:
output1 = fuzzify_layer(train_data)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
output1

tensor([[[8.0841e-01, 6.3008e-01, 1.0181e-29],
         [8.8600e-01, 5.6443e-02, 9.1978e-01]],

        [[9.4973e-01, 0.0000e+00, 1.9082e-07],
         [9.6050e-01, 7.6670e-02, 9.2850e-01]],

        [[9.9999e-01, 0.0000e+00, 9.4824e-01],
         [7.7268e-01, 2.6317e-01, 9.6398e-01]],

        [[9.2577e-01, 7.4599e-30, 6.1043e-11],
         [6.9037e-01, 3.1439e-02, 9.0329e-01]],

        [[8.5725e-01, 5.0265e-06, 1.4599e-21],
         [5.7488e-01, 2.2716e-02, 8.9422e-01]]])

###1.1.O Proposal 1 Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [17]:
def gaussian_3a(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x.unsqueeze(x.dim()) - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [18]:
def gaussian_3b(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

#input is required to be x.unsqueeze(x.dim())

In [19]:
uniq_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [20]:
premises = torch.tensor([[[10, 20, 1],
                          [30, 40, 2]],

                         [[50, 60, 3],
                          [70, 80, 4]]])

In [21]:
gaussian_3a(uniq_tensor, premises)

tensor([[0.9037, 0.5912],
        [0.3829, 0.2357]])

In [22]:
gaussian_3a(x_multi, premises)

tensor([[[0.9037, 0.5912],
         [0.3829, 0.2357]],

        [[0.9231, 0.6126],
         [0.4141, 0.2563]],

        [[0.9406, 0.6341],
         [0.4463, 0.2780]]])

De esta manera es posible definir diversas funciones para usarlas como MFs:

In [23]:
uniq_tensor = torch.tensor([0.4681, 0.3757])
uniq_tensor

tensor([0.4681, 0.3757])

In [24]:
x_multi = torch.tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])
x_multi

tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])

In [25]:
#For functions with 4 parameters
premises4 = torch.rand(2, 2, 4)
premises4

tensor([[[0.3971, 0.8801, 0.8732, 0.8796],
         [0.9776, 0.7342, 0.6896, 0.6147]],

        [[0.6010, 0.6194, 0.0633, 0.8268],
         [0.6363, 0.1963, 0.6919, 0.1181]]])

In [26]:
#For functions with 3 parameters
premises3 = torch.rand(2, 2, 3)
premises3

tensor([[[0.5039, 0.3839, 0.3257],
         [0.9042, 0.5744, 0.2995]],

        [[0.2513, 0.3997, 0.4338],
         [0.3252, 0.8883, 0.4862]]])

In [27]:
#For functions with 2 parameters
premises2 = torch.rand(2, 2, 2)
premises2

tensor([[[0.7549, 0.9284],
         [0.6625, 0.1681]],

        [[0.8026, 0.8271],
         [0.3039, 0.9545]]])

In [28]:
#INPUT x WILL BE CONSIDERED TO ALWAYS BE: x.unsqueeze(x.dim())

In [29]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [30]:
gaussian2(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[1.0489, 1.9519],
        [1.1425, 1.0028]])

In [31]:
gaussian2(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[1.0489, 1.9519],
         [1.1425, 1.0028]],

        [[1.0028, 1.0090],
         [1.3868, 1.0160]],

        [[1.0133, 2.8463],
         [1.2045, 1.0000]]])

In [32]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [33]:
gaussian3(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.9986, 0.9173],
        [0.9792, 0.9992]])

In [34]:
gaussian3(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.9986, 0.9173],
         [0.9792, 0.9992]],

        [[0.9644, 0.9784],
         [0.9814, 0.9888]],

        [[0.8366, 1.0000],
         [0.9970, 0.9998]]])

In [35]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [36]:
bell(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.7252, 0.8732],
        [0.7632, 0.8720]])

In [37]:
bell(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.7252, 0.8732],
         [0.7632, 0.8720]],

        [[0.5645, 0.7270],
         [0.4646, 0.4645]],

        [[0.4730, 0.6129],
         [0.6206, 0.7258]]])

In [38]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [39]:
triangular(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[0.3902, 0.5472],
        [0.4377, 0.0000]])

In [40]:
triangular(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[0.3902, 0.5472],
         [0.4377, 0.0000]],

        [[0.6775, 0.2198],
         [0.1363, 0.0000]],

        [[0.9699, 0.0000],
         [0.3410, 0.0000]]])

In [41]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

In [42]:
trapezoidal(uniq_tensor.unsqueeze(unic_tensor.dim()), premises4)

tensor([[0.1470, 0.0000],
        [0.0000, 0.4489]])

In [43]:
trapezoidal(x_multi.unsqueeze(x_multi.dim()), premises4)

tensor([[[0.1470, 0.0000],
         [0.0000, 0.4489]],

        [[0.5961, 0.9387],
         [0.0000, 0.0273]],

        [[0.0000, 0.2956],
         [0.0000, 0.3137]]])

Ahora ya se tienen funciones que pueden usarse como MFs flexibles (permiten operaciones sobre un dato de entrada solo y también sobre batches de estos mismos)

In [44]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_shape : dimension of the input data
        features : number of initial features
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_shape, features=3, mf=gaussian3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.input_shape = input_shape
        self.features = features
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(input_shape[-1], features, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(input_shape[-1], features, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_shape[-1]):
            print(f"    x{i} parameters:")
            for j in range(self.features):
                print(f"        feature {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [45]:
layer1_0 = FuzzifyLayer(x_multi.shape)

In [46]:
print("Premises: ")
layer1_0.premises

Premises: 


tensor([[[0.9006, 0.6402, 0.8828],
         [0.8723, 0.8676, 0.3259],
         [0.3766, 0.9437, 0.8036]],

        [[0.4501, 0.3691, 0.0843],
         [0.7147, 0.0441, 0.5904],
         [0.0427, 0.3979, 0.6032]]])

In [47]:
layer1_0.premises_structure

Premises Structure:
    x0 parameters:
        feature 1:
            sigma: 0.9005794525146484
            mu: 0.64021235704422
            f: 0.8828015327453613
        feature 2:
            sigma: 0.8723044991493225
            mu: 0.8675967454910278
            f: 0.32591354846954346
        feature 3:
            sigma: 0.376578688621521
            mu: 0.9436739087104797
            f: 0.803623616695404
    x1 parameters:
        feature 1:
            sigma: 0.45005232095718384
            mu: 0.36912739276885986
            f: 0.08427774906158447
        feature 2:
            sigma: 0.714685320854187
            mu: 0.04406160116195679
            f: 0.5904259085655212
        feature 3:
            sigma: 0.04269367456436157
            mu: 0.3978552222251892
            f: 0.6031611561775208


In [48]:
batch_data0 = torch.rand(5,2)
batch_data0

tensor([[0.6761, 0.0938],
        [0.3171, 0.5099],
        [0.4341, 0.3042],
        [0.1493, 0.0439],
        [0.7483, 0.3395]])

In [49]:
single_data0 = torch.rand(2)
single_data0

tensor([0.7746, 0.5010])

In [50]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[9.4718e-01, 9.9170e-01, 9.6033e-01],
         [9.6151e-01, 3.4733e-26, 9.9504e-01]],

        [[6.9306e-01, 9.3545e-01, 9.9841e-01],
         [9.9889e-01, 1.6978e-03, 6.5979e-01]],

        [[7.9108e-01, 9.5928e-01, 9.9851e-01],
         [9.9344e-01, 7.4280e-12, 8.7787e-01]],

        [[5.4450e-01, 8.9299e-01, 9.7696e-01],
         [9.5027e-01, 1.9355e-30, 1.0000e+00]],

        [[9.7534e-01, 9.9668e-01, 9.3955e-01],
         [9.9622e-01, 5.0314e-10, 8.4553e-01]]])

In [51]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[9.8305e-01, 9.9793e-01, 9.3102e-01],
        [9.9920e-01, 9.6639e-04, 6.7017e-01]])

###1.1.2 Proposal 1 with initialization of MATLAB premises

In [51]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_shape : dimension of the input data
        features : number of initial features
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_shape, features=3, mf=gaussian3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.input_shape = input_shape
        self.features = features
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(input_shape[-1], features, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(input_shape[-1], features, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_shape[-1]):
            print(f"    x{i} parameters:")
            for j in range(self.features):
                print(f"        feature {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

###1.P Some things to consider
* **COULD IT BE BETTER TO DEFINE THE GAUSSIAN FUNCTION WITHIN THE LAYER?** (I don't think so, the model would lose flexibility)
* **MATLAB PREMISES INITIALIZATION STILL MUST BE APPLIED**

###1.2 Proposal 2

In [52]:
#STILL NOTHING

##2. Middle Layers

###2.1 Generalized "AND" Layer

In [53]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [54]:
result = multiplication_and(output1)
result

tensor([[7.1625e-01, 3.5564e-02, 9.3639e-30],
        [9.1222e-01, 0.0000e+00, 1.7718e-07],
        [7.7268e-01, 0.0000e+00, 9.1409e-01],
        [6.3912e-01, 2.3453e-31, 5.5140e-11],
        [4.9282e-01, 1.1418e-07, 1.3055e-21]])

In [55]:
class GeneralizedANDLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS
    '''
    def __init__(self):
        super(GeneralizedANDLayer, self).__init__()

    def forward(self, x):
        return x.transpose(1, 2).prod(dim=2)

In [56]:
class GeneralizedANDLayer2(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Propertues:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(GeneralizedANDLayer2, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return self.and_function(x)

In [57]:
generalizedAND_layer = GeneralizedANDLayer2()

In [58]:
output2 = generalizedAND_layer(output1)
'''
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
output2

tensor([[7.1625e-01, 3.5564e-02, 9.3639e-30],
        [9.1222e-01, 0.0000e+00, 1.7718e-07],
        [7.7268e-01, 0.0000e+00, 9.1409e-01],
        [6.3912e-01, 2.3453e-31, 5.5140e-11],
        [4.9282e-01, 1.1418e-07, 1.3055e-21]])

###2.2 Normalization Layer


In [59]:
tensor_input = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
normalized_tensor = torch.nn.functional.normalize(tensor_input)
print(normalized_tensor)

tensor([[0.2673, 0.5345, 0.8018],
        [0.4558, 0.5698, 0.6838]])


In [60]:
class NormalizationLayer(nn.Module):
    '''
    Class that represents the third layer of an ANFIS
    '''
    def __init__(self):
        super(NormalizationLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x)

In [61]:
normalization_layer = NormalizationLayer()

In [62]:
output3 = normalization_layer(output2)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output3

tensor([[9.9877e-01, 4.9592e-02, 1.3057e-29],
        [1.0000e+00, 0.0000e+00, 1.9423e-07],
        [6.4556e-01, 0.0000e+00, 7.6371e-01],
        [1.0000e+00, 3.6696e-31, 8.6274e-11],
        [1.0000e+00, 2.3169e-07, 2.6490e-21]])

###2.3 Generalized "AND" + Normalization Layer (Normalized Firing levels layer)
The idea is to do the AND and normalization operation in a single layer (since there are no trainable parameters here), in this way the outputs would be the already normalized firing levels

In [63]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [64]:
class FiringLevelsLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Attributes:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(FiringLevelsLayer, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return torch.nn.functional.normalize(self.and_function(x))

In [65]:
firing_levels_layer = FiringLevelsLayer()

In [66]:
output23 = firing_levels_layer(output1)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output23

tensor([[9.9877e-01, 4.9592e-02, 1.3057e-29],
        [1.0000e+00, 0.0000e+00, 1.9423e-07],
        [6.4556e-01, 0.0000e+00, 7.6371e-01],
        [1.0000e+00, 3.6696e-31, 8.6274e-11],
        [1.0000e+00, 2.3169e-07, 2.6490e-21]])

###2.3.O Normalized Firing Levels Layer Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [67]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[9.8305e-01, 9.9793e-01, 9.3102e-01],
        [9.9920e-01, 9.6639e-04, 6.7017e-01]])

In [68]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[9.4718e-01, 9.9170e-01, 9.6033e-01],
         [9.6151e-01, 3.4733e-26, 9.9504e-01]],

        [[6.9306e-01, 9.3545e-01, 9.9841e-01],
         [9.9889e-01, 1.6978e-03, 6.5979e-01]],

        [[7.9108e-01, 9.5928e-01, 9.9851e-01],
         [9.9344e-01, 7.4280e-12, 8.7787e-01]],

        [[5.4450e-01, 8.9299e-01, 9.7696e-01],
         [9.5027e-01, 1.9355e-30, 1.0000e+00]],

        [[9.7534e-01, 9.9668e-01, 9.3955e-01],
         [9.9622e-01, 5.0314e-10, 8.4553e-01]]])

In [69]:
batch_output1_0.dim()

3

In [70]:
single_output1_0.dim()

2

In [71]:
def multiplication_and(x):
    return x.prod(dim=x.dim()-2)

In [72]:
batch_firing_levels = multiplication_and(batch_output1_0)
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
batch_firing_levels

tensor([[9.1072e-01, 3.4445e-26, 9.5556e-01],
        [6.9230e-01, 1.5882e-03, 6.5873e-01],
        [7.8589e-01, 7.1255e-12, 8.7656e-01],
        [5.1742e-01, 1.7284e-30, 9.7695e-01],
        [9.7166e-01, 5.0146e-10, 7.9442e-01]])

In [73]:
batch_firing_levels.dim()

2

In [74]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=-1)

tensor([[4.8799e-01, 1.8457e-26, 5.1201e-01],
        [5.1182e-01, 1.1741e-03, 4.8701e-01],
        [4.7273e-01, 4.2862e-12, 5.2727e-01],
        [3.4625e-01, 1.1566e-30, 6.5375e-01],
        [5.5018e-01, 2.8394e-10, 4.4982e-01]])

In [75]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=1)

tensor([[4.8799e-01, 1.8457e-26, 5.1201e-01],
        [5.1182e-01, 1.1741e-03, 4.8701e-01],
        [4.7273e-01, 4.2862e-12, 5.2727e-01],
        [3.4625e-01, 1.1566e-30, 6.5375e-01],
        [5.5018e-01, 2.8394e-10, 4.4982e-01]])

In [76]:
single_firing_levels = multiplication_and(single_output1_0)
''' wi = feature [i] firing level
[w1, w2, w3]
'''
single_firing_levels

tensor([9.8226e-01, 9.6440e-04, 6.2394e-01])

In [77]:
#normalization should result: [0.3792, 0.2889, 0.3318]

In [78]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=0)

tensor([6.1118e-01, 6.0006e-04, 3.8822e-01])

In [79]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=-1)

tensor([6.1118e-01, 6.0006e-04, 3.8822e-01])

In [80]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self, ):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [81]:
layer23_0 = FiringLevelsLayer()

In [82]:
single_normalized_FL = layer23_0(single_output1_0)
''' ~wi = normalization(wi)
    wi = feature [i] firing level
[~w1, ~w2, ~w3]
'''
single_normalized_FL

tensor([6.1118e-01, 6.0006e-04, 3.8822e-01])

In [83]:
batch_normalized_FL = layer23_0(batch_output1_0)
''' ~w = normalization(w)
    wi = feature [i] firing level
[[~w1, ~w2, ~w3],  (batch input1)
 [~w1, ~w2, ~w3],  (batch input2)
 ...
 [~w1, ~w2, ~w3]]  (last batch input)
'''
batch_normalized_FL

tensor([[4.8799e-01, 1.8457e-26, 5.1201e-01],
        [5.1182e-01, 1.1741e-03, 4.8701e-01],
        [4.7273e-01, 4.2862e-12, 5.2727e-01],
        [3.4625e-01, 1.1566e-30, 6.5375e-01],
        [5.5018e-01, 2.8394e-10, 4.4982e-01]])

###2.P Some things to consider
* **IT SHOULD BE MORE OPTIMAL TO PERFORM THE NORMALIZATION IN THE CONSEQUENT PART, that is, when the w is multiplied with the consequent function** (apparently it will be the same, however it is likely that using pytorch's functional.normalize is better )
* **WOULD IT BE USEFUL TO EXTRACT/SHOW THE FIRING LEVELS ON SCREEN WITHOUT NORMALIZING??**(should be possible in the anfis class)
* ** MUST VERIFY THAT THE NORMALIZATION USED HERE IS THE SAME ONE APPLIED IN MATLAB, the one in this jupyter notebook is a simple normalization of the type: [1, 2] -> [1/(1+2), 2/ (1+2)], the use of any exponent or special function in matlab has not been verified**

##3. Consequent Layer

###3.1 Proposal 1

In [84]:
x = torch.tensor([[1, 2],
                  [3, 4]], dtype=torch.float32)
'''
[[x1, x2],  (batch input1)
 [x1, x2]]  (batch input2)
'''

c = torch.tensor([[10, 20, 30],
                  [40, 50, 60]], dtype=torch.float32)
'''
[[p1, q1, r1],  (feature 1 consequent parameters)
 [p2, q2, r2]]  (feature 2 consequent parameters)
'''

w = torch.tensor([[100, 200],
                  [300, 400]], dtype=torch.float32)
'''
[[~w1, ~w2],  (batch input 1 normalized firing levels)
 [~w1, ~w2]]  (batch input 2 normalized firing levels)
'''

'\n[[~w1, ~w2],  (batch input 1 normalized firing levels)\n [~w1, ~w2]]  (batch input 2 normalized firing levels)\n'

In [85]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [86]:
'''
[[x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2],  (x1, x2 of batch input1)
 [x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2]]  (x1, x2 of batch input2)
'''
simple_linear(x, c)

tensor([[ 80., 200.],
        [140., 380.]])

In [87]:
def typical_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)).mul(w)

In [88]:
'''
[[~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)],  (x1, x2, ~w1 y ~w2 of batch input1)
 [~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)]]  (x1, x2, ~w1 y ~w2 of batch input2)
'''
typical_linear(x, c, w)

tensor([[  8000.,  40000.],
        [ 42000., 152000.]])

In [89]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        data_shape : dimension of the input data
        features : number of features
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, features=3, function=typical_linear):
        super(ConsequentLayer, self).__init__()
        self.features = features
        self.data_shape = data_shape
        self.function = function
        #CONSEQUENT PARAMETERS inicializados aleatoriamente de momento
        self.consequents = torch.rand(features, data_shape[1] + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    def show_consequents(self):
        print("Consequents tensor:")
        print(self.consequents)

    def consequents(self):
        return self.consequents

In [90]:
consequent_layer = ConsequentLayer(train_data.shape)

In [91]:
# x : input data (for train)
train_data

tensor([[0.2105, 0.8656],
        [0.5940, 0.8380],
        [0.9710, 0.7050],
        [0.5111, 0.9146],
        [0.3231, 0.9400]])

In [92]:
# c : consequent parameters by feature
consequent_layer.consequents

tensor([[0.3282, 0.2799, 0.1708],
        [0.9185, 0.4408, 0.2604],
        [0.8584, 0.5532, 0.8913]])

In [93]:
# w: normalized firing levels for input on batch
output23

tensor([[9.9877e-01, 4.9592e-02, 1.3057e-29],
        [1.0000e+00, 0.0000e+00, 1.9423e-07],
        [6.4556e-01, 0.0000e+00, 7.6371e-01],
        [1.0000e+00, 3.6696e-31, 8.6274e-11],
        [1.0000e+00, 2.3169e-07, 2.6490e-21]])

In [94]:
output4 = consequent_layer(train_data, output23)
''' ~w = normalization(w)   --> normalized firing levels
    fi = x1*pi + x2*qi + ri --> consequent function of feature i applied to an input (x1, x2)

[[~w1*f1, ~w2*f2, ~w3*f3],  (of batch input1)
 [~w1*f1, ~w2*f2, ~w3*f3],   (of batch input2)
 ...
 [~w1*f1, ~w2*f2, ~w3*f3]]   (of batch last input)
'''
output4

tensor([[4.8153e-01, 4.1428e-02, 2.0250e-29],
        [6.0027e-01, 0.0000e+00, 3.6220e-07],
        [4.4335e-01, 0.0000e+00, 1.6151e+00],
        [5.9448e-01, 4.1579e-31, 1.5840e-10],
        [5.3989e-01, 2.2511e-07, 4.4733e-21]])

###3.1.O Proposal 1 Optimized and Flexible

In [95]:
single_normalized_FL

tensor([6.1118e-01, 6.0006e-04, 3.8822e-01])

In [96]:
batch_normalized_FL

tensor([[4.8799e-01, 1.8457e-26, 5.1201e-01],
        [5.1182e-01, 1.1741e-03, 4.8701e-01],
        [4.7273e-01, 4.2862e-12, 5.2727e-01],
        [3.4625e-01, 1.1566e-30, 6.5375e-01],
        [5.5018e-01, 2.8394e-10, 4.4982e-01]])

In [97]:
single_data0

tensor([0.7746, 0.5010])

In [98]:
batch_data0

tensor([[0.6761, 0.0938],
        [0.3171, 0.5099],
        [0.4341, 0.3042],
        [0.1493, 0.0439],
        [0.7483, 0.3395]])

In [99]:
params = ["p", "q", "r"]
#the number of parameters depends on the dimension of the input data

consequents = torch.rand(layer1_0.features, len(params))
'''
[[p1, q1, r1],  (consequent parameters of feature 1)
 [p2, q2, r2]]  (consequent parameters of feature 2)
 [p3, q3, r3]]  (consequent parameters of feature 3)
'''
consequents

tensor([[0.0711, 0.3997, 0.2173],
        [0.6581, 0.4198, 0.5880],
        [0.3688, 0.3804, 0.2438]])

In [100]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [101]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[[f1(x1, x2), f2(x1, x2), f3(x1, x2)],  ((x1, x2) of batch input1)
 [f1(x1, x2), f2(x1, x2), f3(x1, x2)]]  ((x1, x2) of batch input2)
'''
simple_linear(batch_data0, consequents)

tensor([[0.3029, 1.0724, 0.5289],
        [0.4437, 1.0108, 0.5547],
        [0.3698, 1.0014, 0.5196],
        [0.2455, 0.7047, 0.3156],
        [0.4062, 1.2230, 0.6489]])

In [102]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[f1(x1, x2), f2(x1, x2), f3(x1, x2)]  ((x1, x2) the input values)
'''
simple_linear(single_data0, consequents)

tensor([[0.4726, 1.3081, 0.7201]])

In [103]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

In [104]:
weighted_linear(single_data0, consequents, single_normalized_FL)

tensor([0.2889, 0.0008, 0.2796])

In [105]:
weighted_linear(batch_data0, consequents, batch_normalized_FL)

tensor([[1.4780e-01, 1.9792e-26, 2.7078e-01],
        [2.2707e-01, 1.1868e-03, 2.7016e-01],
        [1.7480e-01, 4.2922e-12, 2.7399e-01],
        [8.5003e-02, 8.1506e-31, 2.0633e-01],
        [2.2348e-01, 3.4726e-10, 2.9191e-01]])

In [106]:
single_data0.shape

torch.Size([2])

In [107]:
single_data0.shape[-1]

2

In [108]:
batch_data0.shape

torch.Size([5, 2])

In [109]:
single_data0.shape[-1]

2

In [110]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        input_shape : dimension of the input data
        features : number of features
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_shape, features=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.features = features
        self.input_shape = input_shape
        self.function = function
        #CONSEQUENT PARAMETERS randomly initialized at the moment
        self.consequents = torch.rand(features, input_shape[-1] + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.features):
            print(f"    feature {i + 1} consequent parameters: {self.consequents[i]}")

In [111]:
layer4_0 = ConsequentLayer(single_data0.shape)

In [112]:
single_data0

tensor([0.7746, 0.5010])

In [113]:
single_normalized_FL

tensor([6.1118e-01, 6.0006e-04, 3.8822e-01])

In [114]:
layer4_0.consequents_structure

Consequents Structure:
    feature 1 consequent parameters: tensor([0.7012, 0.6012, 0.2268])
    feature 2 consequent parameters: tensor([0.9922, 0.3022, 0.4539])
    feature 3 consequent parameters: tensor([0.4839, 0.0744, 0.6447])


In [115]:
single_ponderated_consequents = layer4_0(single_data0, single_normalized_FL)
single_ponderated_consequents

tensor([0.6547, 0.0008, 0.4103])

In [116]:
batch_ponderated_consequents = layer4_0(batch_data0, batch_normalized_FL)
batch_ponderated_consequents

tensor([[3.6956e-01, 2.1283e-26, 5.0116e-01],
        [3.8679e-01, 1.0833e-03, 4.0717e-01],
        [3.3757e-01, 4.1858e-12, 4.6261e-01],
        [1.2392e-01, 7.1166e-31, 4.7082e-01],
        [5.2578e-01, 3.6885e-10, 4.6423e-01]])

##4. Output Layer

###4.1 Proposal 1

In [117]:
def sum(x):
    return torch.sum(x, dim=-1)

In [118]:
sum(single_ponderated_consequents)

tensor(1.0658)

In [119]:
sum(batch_ponderated_consequents)

tensor([0.8707, 0.7950, 0.8002, 0.5947, 0.9900])

In [120]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##5. ANFIS

###5.1 Proposal 1

In [121]:
class Type3ANFIS(nn.Module):
    def __init__(self, input_shape, initial_features=3):
        super(Type3ANFIS, self).__init__()
        self.layer1 = FuzzifyLayer(input_shape, features=initial_features)
        self.layer23 = FiringLevelsLayer()
        self.layer4 = ConsequentLayer(input_shape, features=initial_features)
        self.layer5 = OutputLayer()

    def forward(self, x):
        output = self.layer1(x)
        output = self.layer4(x, self.layer23(output))
        output = self.layer5(x)
        return output

In [122]:
anfis1 = Type3ANFIS(train_data.shape)

In [123]:
train_data = torch.rand(5,2)
train_data

tensor([[0.5850, 0.3359],
        [0.4062, 0.8605],
        [0.5930, 0.6550],
        [0.7592, 0.1791],
        [0.7237, 0.0607]])

In [124]:
single_input = train_data[0]
single_input

tensor([0.5850, 0.3359])

In [125]:
batch_final_output = anfis1(train_data)
batch_final_output

tensor([0.9210, 1.2666, 1.2480, 0.9383, 0.7844])

In [126]:
single_final_output = anfis1(single_input)
single_final_output

tensor(0.9210)